In [ ]:
import pandas as pd
import numpy as np

def predict_soh(csv_path, soh_n, qn, q_max, delta_u, a):
    """
    基于电池老化数据预测健康状态
    
    输入:
        csv_path: CSV文件路径
        soh_n: 当前健康状态 (0-1)
        qn: 拟充/放电容量
        q_max: 额定容量 (100% SOH)
        delta_u: 电压变化量
        a: 系数
    
    输出:
        soh_pre: 预测的健康状态
    """
    df = pd.read_csv(csv_path)      # 读取CSV数据，CSV有SoH和N两列
    df = df.sort_values(by='N').reset_index(drop=True)
    
    # 查找SoH_n对应的循环圈数N
    if soh_n in df['SoH'].values:
        # 精确查找
        n = df[df['SoH'] == soh_n]['N'].values[0]
        idx = df[df['SoH'] == soh_n].index[0]
    else:
        # 插值查找：找到SoH_n在两个SoH值之间的位置，使用线性插值反推N
        for i in range(len(df) - 1):
            soh_low = df.loc[i, 'SoH']
            soh_high = df.loc[i + 1, 'SoH']
            
            if (soh_low >= soh_n >= soh_high) or (soh_low <= soh_n <= soh_high):
                n_low = df.loc[i, 'N']
                n_high = df.loc[i + 1, 'N']
                ratio = (soh_n - soh_low) / (soh_high - soh_low)
                n = n_low + ratio * (n_high - n_low)
                idx = i + ratio
                break
        else:
            # 超出范围，使用边界值
            if soh_n > df['SoH'].max():
                n = df['N'].min()
            else:
                n = df['N'].max()
            idx = None
    
    # 获取N+1对应的SoH_n+1
    if isinstance(idx, (int, np.integer)):
        # 精确查找
        if idx + 1 < len(df):
            soh_n_plus_1 = df.loc[idx + 1, 'SoH']
            n_plus_1 = df.loc[idx + 1, 'N']
        else:
            # 已经是最后一行，使用外推或相同值
            soh_n_plus_1 = df.loc[idx, 'SoH']
            n_plus_1 = df.loc[idx, 'N']
    else:
        # 插值查找
        idx_int = int(idx)
        idx_frac = idx - idx_int
        
        if idx_int + 1 < len(df):
            # 插值计算SoH_n+1
            soh_next_low = df.loc[idx_int + 1, 'SoH']
            if idx_int + 2 < len(df):
                soh_next_high = df.loc[idx_int + 2, 'SoH']
                soh_n_plus_1 = soh_next_low + idx_frac * (soh_next_high - soh_next_low)
            else:
                soh_n_plus_1 = soh_next_low
        else:
            soh_n_plus_1 = df.loc[idx_int, 'SoH']
    
    # 计算等效放电圈数 η
    soh_t = soh_n
    eta = abs((qn / (q_max * soh_t)) * (delta_u / a))        # η = (Qn / (Q_max * SoH_t)) * (delta_U / A)
    '''这里的delta_u是同一圈循环中充电或放电时PCS端电压的变化量, 若系统提供, 则可同步计算出系数a, 若不提供, 则令delta_u与a为常数'''
    eta = max(0.0, min(1.0, eta))
    
    # 预测SoH
    soh_pre = (1 - eta) * soh_n + eta * soh_n_plus_1     # SoH_pre = (1-η) * SoH_n + η * SoH_n+1
    
    return {
        'soh_pre': soh_pre,
    }

# ==================== 示例 ====================

if __name__ == "__main__":
    
    # 模拟电池老化数据，随着循环圈数增加，SoH下降
    test_data = {
        'N': [1, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],
        'SoH': [1.0, 0.98, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]
    }
    test_df = pd.DataFrame(test_data)
    test_df.to_csv('age.csv', index=False)

    # 输入参数
    soh_n = 0.92      # 当前健康状态
    qn = 2.0          # 拟充/放电容量 (Ah)
    q_max = 3.0       # 额定容量(100% SOH, Ah)
    '''这里的delta_u是同一圈循环中充电或放电时PCS端电压的变化量, 若系统提供, 则可同步计算出系数a, 若不提供, 则令delta_u = 0.1, a = 1.0'''
    delta_u = 0.1     # 电压变化量 (V)
    a = 1.0           # 系数
    
    # 执行预测
    result = predict_soh('age.csv', soh_n, qn, q_max, delta_u, a)
    
    # 输出结果
    print("预测结果：")
    print(f"输入:\nSoH_n: {soh_n}\nQ_n：{qn}\n")
    print(f"预测SoH_pre: {result['soh_pre']:.4f}")

预测结果：
输入:
SoH_n: 0.92
Q_n：2.0

预测SoH_pre: 0.9164
